# 10 — Prototyp: ansiktsblocklista (t.ex. Hitler)
Istället för att träna en klassificerare används ansikts-**embeddings** via FaceNet (samma `facenet_pytorch`-bibliotek som redan används för MTCNN-croppning i appen). Ett ansikte omvandlas till en numerisk vektor, och nya uppladdningar jämförs mot ett litet set referensembeddings.

**Innan du kör:** lägg 3-5 olika bilder av personen att blockera i `data/blocklist_reference/hitler/` (redan skapad).

In [ ]:
import os
import glob
import numpy as np
import torch
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1

# FaceNet förväntar sig 160x160 (annan storlek än mustasch-modellernas 178x178)
mtcnn_face = MTCNN(image_size=160, margin=0, post_process=True)
facenet = InceptionResnetV1(pretrained='vggface2').eval()

print('FaceNet-modell laddad (vggface2-vikter).')

## Helper-funktioner

In [ ]:
def get_embedding(image):
    """Tar en PIL-bild, returnerar en 512-dimensionell embedding eller None om inget ansikte hittas."""
    image = image.convert('RGB')
    face = mtcnn_face(image)
    if face is None:
        return None
    with torch.no_grad():
        embedding = facenet(face.unsqueeze(0))
    return embedding.squeeze(0).numpy()


def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def build_reference_embeddings(folder):
    """Returnerar (paths, embeddings) som garanterat matchande, parallella listor —
    annars kan två separata glob()-anrop råka returnera olika ordning och allt glider isär."""
    all_paths = glob.glob(os.path.join(folder, '*.jpg')) + \
                glob.glob(os.path.join(folder, '*.jpeg')) + \
                glob.glob(os.path.join(folder, '*.png'))

    used_paths = []
    embeddings = []

    for path in all_paths:
        emb = get_embedding(Image.open(path))
        if emb is not None:
            used_paths.append(path)
            embeddings.append(emb)
        else:
            print(f'Inget ansikte hittat i: {path}')

    print(f'{len(embeddings)} referensembeddings byggda från {len(all_paths)} bilder.')
    return used_paths, embeddings


def is_blocked(image, reference_embeddings, threshold=0.6):
    """Returnerar (blockerad: bool, högsta_likhet: float)."""
    emb = get_embedding(image)
    if emb is None:
        return False, 0.0

    similarities = [cosine_similarity(emb, ref) for ref in reference_embeddings]
    best = max(similarities) if similarities else 0.0
    return best >= threshold, best

## Bygg referensembeddings
Lägg 3-5 bilder i `data/blocklist_reference/hitler/` innan du körkör denna.

In [ ]:
REFERENCE_DIR = 'data/blocklist_reference/hitler'
ref_paths, reference_embeddings = build_reference_embeddings(REFERENCE_DIR)

## Testa tröskelvärdet
Testa mot dina referensbilder (bör ge hög likhet, nära 1.0) och mot vanliga ansikten från `reference_test_images` (bör ge låg likhet). Justera `threshold` i `is_blocked()` baserat på vad du ser här.

In [ ]:
import pandas as pd

rows = []

# Testa referensbilderna mot varandra (cross-check) — bör ge hög likhet
for path in ref_paths:
    blocked, similarity = is_blocked(Image.open(path), reference_embeddings, threshold=0.6)
    rows.append({'bild': os.path.basename(path), 'typ': 'referens (bör blockeras)', 'likhet': round(similarity, 3), 'blockerad': blocked})

# Testa mot vanliga testbilder — bör INTE blockeras
normal_paths = glob.glob('data/reference_test_images/*')[:15]
for path in normal_paths:
    blocked, similarity = is_blocked(Image.open(path), reference_embeddings, threshold=0.6)
    rows.append({'bild': os.path.basename(path), 'typ': 'vanlig (bör INTE blockeras)', 'likhet': round(similarity, 3), 'blockerad': blocked})

df = pd.DataFrame(rows).sort_values('likhet', ascending=False)
df

## Leave-one-out-test (det ärliga testet)
Förra cellen läckte — varje referensbild jämfördes delvis mot sig själv (1.000 är meningslöst). Här jämförs varje referensbild bara mot **de andra nio**, så vi ser om olika foton av samma person genuint känns igen som samma person.

In [ ]:
loo_rows = []

for i, path in enumerate(ref_paths):
    emb = get_embedding(Image.open(path))
    if emb is None:
        continue

    others = [reference_embeddings[j] for j in range(len(reference_embeddings)) if j != i]
    similarities = [cosine_similarity(emb, ref) for ref in others]
    best = max(similarities) if similarities else 0.0

    loo_rows.append({
        'bild': os.path.basename(path),
        'bäst_likhet_mot_övriga_referenser': round(best, 6),
        'blockerad_vid_0.6': best >= 0.6
    })

loo_df = pd.DataFrame(loo_rows).sort_values('bäst_likhet_mot_övriga_referenser', ascending=False)
loo_df

In [ ]:
# Diagnos: är det MTCNN-croppen själv som blir identisk, eller embeddingen?
path_a, path_b = ref_paths[0], ref_paths[1]
print('Jämför:', os.path.basename(path_a), 'vs', os.path.basename(path_b))

face_a = mtcnn_face(Image.open(path_a).convert('RGB'))
face_b = mtcnn_face(Image.open(path_b).convert('RGB'))

if face_a is None or face_b is None:
    print('MTCNN hittade inget ansikte i en av bilderna — det är boven.')
else:
    pixel_diff = (face_a - face_b).abs().sum().item()
    print(f'Total pixelskillnad mellan croppade ansikten: {pixel_diff}')
    print('(0.0 betyder att MTCNN-croppen blev identisk för två olika foton)')

    emb_a = get_embedding(Image.open(path_a))
    emb_b = get_embedding(Image.open(path_b))
    embedding_diff = float(np.abs(emb_a - emb_b).sum())
    print(f'Total skillnad mellan embeddings: {embedding_diff}')
    print('(0.0 betyder att embeddingen blev identisk trots olika input)')

## Nästa steg
Om leave-one-out-resultaten ovan **fortfarande** ligger klart över tröskeln (0.6) och separerade från de "vanliga" bilderna i förra tabellen — då är det ett genuint, pålitligt resultat. Om de istället ligger lägre (typ 0.4-0.5), behöver tröskeln sänkas något, eller fler/bättre referensbilder läggas till.

När tröskeln känns trygg, flytta in funktionerna `get_embedding`, `cosine_similarity`, `build_reference_embeddings` och `is_blocked` i `app_3class.py` och kör som ett extra filter innan mustasch-analysen, ungefär:

```python
blocked, similarity = is_blocked(image, reference_embeddings, threshold=0.6)
if blocked:
    st.error('Den här personen kan inte analyseras.')
```

Referensembeddings kan räknas en gång vid appstart (`@st.cache_resource`) istället för varje gång, så det inte blir tungt.